# Feature Engineering e Preparacao para Machine Learning
## Pipeline de Risco de Credito

**Autora:** Nayane Araujo  
**GitHub:** [Nayanearaujo](https://github.com/Nayanearaujo)  

---

### O que e Feature Engineering?

Feature Engineering e a arte de transformar dados brutos em informacoes uteis para o modelo de Machine Learning.

Pense assim: um modelo de ML nao entende texto como 'RENT' ou 'OWN'. Ele so entende numeros. E nao basta converter, precisa converter de forma que o significado seja preservado.

Neste notebook vamos:

1. Carregar os dados limpos da camada Silver
2. Criar novas variaveis que o modelo nao teria como descobrir sozinho
3. Codificar variaveis categoricas de forma inteligente
4. Balancear as classes com SMOTE
5. Escalonar as variaveis numericas
6. Analisar a importancia das features com SHAP
7. Preparar o dataset final para treinamento

---

## 1. Importacoes

In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '{:.4f}'.format)

# Adiciona o root do projeto ao path
sys.path.insert(0, str(Path('..').resolve()))

print('Bibliotecas carregadas!')

## 2. Carregamento dos Dados Silver

Aqui carregamos os dados ja limpos da camada Silver. Se ainda nao existirem, rodamos o pipeline automaticamente.

Lembre: a camada Silver ja tem os dados sem nulos, com tipos corretos e sem registros invalidos.

In [ ]:
SILVER_PATH = Path('../data/silver/credit_risk_clean.csv')

if not SILVER_PATH.exists():
    print('Silver nao encontrada. Executando pipeline...')
    from src.transformation.silver_transform import run_silver_transform
    df = run_silver_transform()
else:
    df = pd.read_csv(SILVER_PATH)
    print(f'Silver carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

# Remove colunas de metadados do pipeline
meta_cols = [c for c in df.columns if c.startswith('_')]
df = df.drop(columns=meta_cols)

print(f'Dataset para feature engineering: {df.shape}')
df.head(3)

## 3. Criacao de Variaveis Derivadas

Variaveis derivadas sao novas colunas calculadas a partir das existentes. Elas capturam **relacoes** que o modelo teria dificuldade de descobrir sozinho.

Por exemplo: o valor do emprestimo e a renda, separados, dizem pouco. Mas a razao entre eles (quanto da renda seria usada para pagar o emprestimo) e muito informativa!

Cada variavel nova que criamos deve ter uma justificativa de negocio clara.

In [ ]:
df_feat = df.copy()

# Feature 1: Custo total estimado do emprestimo
# Razao: quanto o cliente vai pagar no total? Quanto maior, maior o compromisso
df_feat['custo_total_estimado'] = df_feat['loan_amnt'] * (1 + df_feat['loan_int_rate'] / 100)

# Feature 2: Renda por ano de emprego
# Razao: mede a estabilidade financeira. Alta renda com pouco tempo = risco maior?
df_feat['renda_por_ano_emprego'] = df_feat['person_income'] / (df_feat['person_emp_length'] + 1)

# Feature 3: Valor do emprestimo em relacao ao historico de credito
# Razao: quem tem historico longo tomando emprestimo pequeno e mais confiavel
df_feat['emprestimo_por_historico'] = df_feat['loan_amnt'] / (df_feat['cb_person_cred_hist_length'] + 1)

# Feature 4: Flag de inadimplencia previa (0/1)
df_feat['inadimplencia_previa'] = (df_feat['cb_person_default_on_file'] == 'Y').astype(int)

# Feature 5: Faixa de comprometimento de renda (categorica ordinal)
df_feat['faixa_comprometimento'] = pd.cut(
    df_feat['loan_percent_income'],
    bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=[0, 1, 2, 3, 4]
).astype(float)

novas_features = ['custo_total_estimado', 'renda_por_ano_emprego',
                  'emprestimo_por_historico', 'inadimplencia_previa', 'faixa_comprometimento']

print('Novas variaveis criadas:')
for f in novas_features:
    print(f'  {f:35s}: {df_feat[f].dtype}')

print(f'\nDataset antes: {df.shape[1]} colunas')
print(f'Dataset depois: {df_feat.shape[1]} colunas')

In [ ]:
# Visualizacao: como as novas features se relacionam com a inadimplencia?
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Custo total por status
for status, cor, nome in [(0, '#2ecc71', 'Adimplente'), (1, '#e74c3c', 'Inadimplente')]:
    axes[0].hist(
        df_feat[df_feat['loan_status'] == status]['custo_total_estimado'].clip(upper=50000),
        bins=40, alpha=0.6, color=cor, label=nome, density=True
    )
axes[0].set_title('Custo Total Estimado', fontweight='bold')
axes[0].set_xlabel('Valor (R$)')
axes[0].legend()

# Renda por ano de emprego
for status, cor, nome in [(0, '#2ecc71', 'Adimplente'), (1, '#e74c3c', 'Inadimplente')]:
    axes[1].hist(
        np.log1p(df_feat[df_feat['loan_status'] == status]['renda_por_ano_emprego']),
        bins=40, alpha=0.6, color=cor, label=nome, density=True
    )
axes[1].set_title('Renda por Ano de Emprego (log)', fontweight='bold')
axes[1].set_xlabel('Log(Renda / Anos de Emprego)')
axes[1].legend()

# Taxa por faixa de comprometimento
taxa_faixa = df_feat.groupby('faixa_comprometimento')['loan_status'].mean() * 100
nomes_faixas = ['0-10%', '10-20%', '20-30%', '30-50%', '50-100%']
axes[2].bar(nomes_faixas[:len(taxa_faixa)], taxa_faixa.values,
            color=plt.cm.RdYlGn_r(taxa_faixa.values / 100))
axes[2].set_title('Taxa de Inadimplencia por Comprometimento', fontweight='bold')
axes[2].set_xlabel('Faixa de Comprometimento de Renda')
axes[2].set_ylabel('Taxa (%)')
for i, v in enumerate(taxa_faixa.values):
    axes[2].text(i, v + 0.3, f'{v:.1f}%', ha='center')

plt.suptitle('Variaveis Derivadas x Inadimplencia', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fe_derived_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Codificacao de Variaveis Categoricas

Modelos de Machine Learning trabalham com numeros. Variaveis como `loan_grade` (A, B, C...) ou `loan_intent` (PERSONAL, EDUCATION...) precisam ser convertidas.

Existem duas tecnicas principais:

**OrdinalEncoder**: converte em numeros mantendo a ordem.  
Usamos para `loan_grade` porque existe uma ordem de risco clara: A (menor) ate G (maior).

**One-Hot Encoding**: cria uma coluna binaria para cada categoria.  
Usamos para `loan_intent` e `person_home_ownership` porque nao ha ordem entre as categorias.

Nao usar OrdinalEncoder em variaveis sem ordem e um erro comum! Por exemplo, codificar PERSONAL=1, EDUCATION=2, MEDICAL=3 implicaria que MEDICAL e tres vezes mais algo que PERSONAL, o que nao faz sentido.

In [ ]:
df_encoded = df_feat.copy()

# OrdinalEncoder para loan_grade (ordem de risco A < B < C < D < E < F < G)
enc_grade = OrdinalEncoder(
    categories=[['A', 'B', 'C', 'D', 'E', 'F', 'G']],
    handle_unknown='use_encoded_value',
    unknown_value=np.nan
)
df_encoded['loan_grade_encoded'] = enc_grade.fit_transform(df_encoded[['loan_grade']])

print('OrdinalEncoder para loan_grade:')
grade_mapping = dict(zip(['A', 'B', 'C', 'D', 'E', 'F', 'G'], range(7)))
for grade, num in grade_mapping.items():
    print(f'  {grade} -> {num}')

# One-Hot Encoding para variaveis sem ordem
print('\nOne-Hot Encoding para loan_intent:')
intent_dummies = pd.get_dummies(df_encoded['loan_intent'], prefix='intent', drop_first=True)
print(f'  Colunas criadas: {list(intent_dummies.columns)}')
df_encoded = pd.concat([df_encoded, intent_dummies], axis=1)

print('\nOne-Hot Encoding para person_home_ownership:')
home_dummies = pd.get_dummies(df_encoded['person_home_ownership'], prefix='home', drop_first=True)
print(f'  Colunas criadas: {list(home_dummies.columns)}')
df_encoded = pd.concat([df_encoded, home_dummies], axis=1)

print(f'\nDataset apos encoding: {df_encoded.shape[1]} colunas')

In [ ]:
# Selecao das colunas finais para o modelo
# Removemos as colunas originais categoricas (substituidas pelo encoding)
# e colunas que nao devem entrar no modelo

remover = [
    'loan_status',           # target, nao e feature
    'loan_grade',            # substituida por loan_grade_encoded
    'loan_intent',           # substituida por intent_*
    'person_home_ownership', # substituida por home_*
    'cb_person_default_on_file',  # substituida por inadimplencia_previa
]

feature_cols = [c for c in df_encoded.columns if c not in remover]
X = df_encoded[feature_cols].fillna(df_encoded[feature_cols].median(numeric_only=True))
y = df_encoded['loan_status']

print(f'Features selecionadas para o modelo ({len(feature_cols)} total):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

print(f'\nTarget: loan_status')
print(f'Distribuicao do target: {y.value_counts().to_dict()}')

## 5. Balanceamento com SMOTE

Nosso dataset tem desbalanceamento: cerca de 78% adimplentes e 22% inadimplentes.

**Por que isso e um problema?**

Um modelo treinado num dataset desbalanceado tende a ignorar a classe minoritaria. Ele aprende que chutar sempre a classe majoritaria da um resultado bom em acuracia. Mas na pratica, erramos exatamente nos casos que mais importam: os inadimplentes!

**O que e SMOTE?**

SMOTE significa Synthetic Minority Over-sampling Technique. Ele cria exemplos *sinteticos* (artificiais) da classe minoritaria (inadimplentes), interpolando entre exemplos reais. Nao e simplesmente duplicar linhas, e criar variantes realistas.

**Regra de ouro:** SMOTE so deve ser aplicado no conjunto de **treino**. Nunca no conjunto de teste! Se aplicarmos no teste, estaremos avaliando o modelo em dados artificiais, o que daria uma falsa impressao de performance.

In [ ]:
# Divisao treino/teste ANTES do SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para teste
    random_state=42,    # seed para reproducibilidade
    stratify=y          # mantem a proporcao de classes em treino e teste
)

print('Divisao treino/teste:')
print(f'  Treino: {len(X_train):,} registros')
print(f'  Teste : {len(X_test):,} registros')

print('\nDistribuicao do target no TREINO (antes do SMOTE):')
print(f'  Adimplente  (0): {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.1f}%)')
print(f'  Inadimplente(1): {(y_train == 1).sum():,} ({(y_train == 1).mean()*100:.1f}%)')

In [ ]:
# Aplicacao do SMOTE SOMENTE no treino
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Distribuicao do target no TREINO (apos SMOTE):')
print(f'  Adimplente  (0): {(y_train_bal == 0).sum():,} ({(y_train_bal == 0).mean()*100:.1f}%)')
print(f'  Inadimplente(1): {(y_train_bal == 1).sum():,} ({(y_train_bal == 1).mean()*100:.1f}%)')
print(f'  Total apos SMOTE: {len(X_train_bal):,}')

print('\nConjunto de TESTE (nao alterado):')
print(f'  Adimplente  (0): {(y_test == 0).sum():,} ({(y_test == 0).mean()*100:.1f}%)')
print(f'  Inadimplente(1): {(y_test == 1).sum():,} ({(y_test == 1).mean()*100:.1f}%)')

# Visualizacao antes e depois
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_data, titulo in [
    (axes[0], y_train, 'Antes do SMOTE (Treino)'),
    (axes[1], y_train_bal, 'Depois do SMOTE (Treino)')
]:
    counts = pd.Series(y_data).value_counts()
    labels = ['Adimplente', 'Inadimplente']
    cores = ['#2ecc71', '#e74c3c']
    bars = ax.bar(labels, [counts.get(0, 0), counts.get(1, 0)], color=cores)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Quantidade')
    for bar, v in zip(bars, [counts.get(0, 0), counts.get(1, 0)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('Efeito do SMOTE no Balanceamento das Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fe_smote_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Escalonamento das Variaveis

Variaveis numericas tem escalas muito diferentes: `person_income` pode ser 100.000, enquanto `loan_int_rate` e 15. Se nao escalonarmos, variaveis com valores maiores dominam o modelo.

Usamos o **StandardScaler**, que transforma cada variavel para ter media 0 e desvio padrao 1. Assim todas ficam na mesma escala.

Regra fundamental: o scaler aprende no treino (`fit_transform`) e apenas aplica no teste (`transform`). Nunca faca `fit` no teste, pois isso seria usar informacao do futuro.

In [ ]:
scaler = StandardScaler()

# Aprende a escala NO TREINO e ja transforma
X_train_scaled = scaler.fit_transform(X_train_bal)

# Apenas transforma o teste (sem fit!)
X_test_scaled = scaler.transform(X_test)

print('Escalonamento concluido!')
print(f'Shape treino: {X_train_scaled.shape}')
print(f'Shape teste : {X_test_scaled.shape}')

# Verificacao: media e desvio padrao apos escalonamento
print('\nVerificacao - media e desvio no treino (devem ser ~0 e ~1):')
print(f'  Media    : {X_train_scaled.mean():.4f}')
print(f'  Desvio   : {X_train_scaled.std():.4f}')

## 7. Importancia Inicial das Features

Antes de partir para o modelo final, vamos usar um Random Forest rapido para estimar quais features sao mais importantes.

Isso nos ajuda a:
- Confirmar se as variaveis derivadas que criamos realmente ajudam
- Identificar variaveis inutils (se houver)
- Ter uma primeira visao do que o modelo vai aprender

In [ ]:
# Random Forest rapido para importancia das features
rf_quick = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X_train_scaled, y_train_bal)

# Importancias ordenadas
importances = pd.Series(rf_quick.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

# Grafico
top_n = 15
top_feat = importances.head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
cores = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))
bars = ax.barh(range(top_n), top_feat.values[::-1], color=cores[::-1])
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feat.index[::-1])
ax.set_xlabel('Importancia (Gini Impurity)')
ax.set_title(f'Top {top_n} Features Mais Importantes (Random Forest)', fontweight='bold')

for bar, val in zip(bars, top_feat.values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../docs/fe_feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features mais importantes:')
for i, (col, val) in enumerate(importances.head(10).items(), 1):
    print(f'  {i:2d}. {col:35s}: {val:.4f}')

In [ ]:
# Salva os dados preparados para uso no proximo notebook
import pickle

MODELS_PATH = Path('../src/models')
MODELS_PATH.mkdir(parents=True, exist_ok=True)

# Salva scaler
with open(MODELS_PATH / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Salva nomes das features
pd.Series(feature_cols).to_csv(MODELS_PATH / 'feature_names.csv', index=False)

# Salva datasets preparados como CSV (para o notebook 03)
SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame(X_train_scaled, columns=feature_cols).assign(loan_status=y_train_bal.values).to_csv(
    Path('../data/silver/train_prepared.csv'), index=False
)
pd.DataFrame(X_test_scaled, columns=feature_cols).assign(loan_status=y_test.values).to_csv(
    Path('../data/silver/test_prepared.csv'), index=False
)

print('Dados preparados salvos com sucesso!')
print(f'  Scaler         : src/models/scaler.pkl')
print(f'  Feature names  : src/models/feature_names.csv')
print(f'  Treino         : data/silver/train_prepared.csv ({len(X_train_scaled):,} linhas)')
print(f'  Teste          : data/silver/test_prepared.csv ({len(X_test_scaled):,} linhas)')

## 8. Resumo e Proximos Passos

O que fizemos neste notebook:

In [ ]:
print('=' * 65)
print('RESUMO DO FEATURE ENGINEERING')
print('=' * 65)

print(f'\nVariaveis originais   : {df.shape[1]}')
print(f'Variaveis derivadas   : 5 novas (custo_total, renda_por_emprego, etc)')
print(f'Variaveis apos encoding: {len(feature_cols)}')

print(f'\nDataset de treino antes do SMOTE : {len(X_train):,}')
print(f'Dataset de treino apos SMOTE    : {len(X_train_bal):,}')
print(f'Dataset de teste (intocado)     : {len(X_test):,}')

print('\nTop 5 features mais importantes (Random Forest inicial):')
for i, (col, val) in enumerate(importances.head(5).items(), 1):
    print(f'  {i}. {col}: {val:.4f}')

print('\nProximos passos:')
print('  -> Notebook 03: Treinar e avaliar Regressao Logistica, RF e XGBoost')
print('  -> Notebook 04: Azure e Databricks')
print('=' * 65)

---

Dados preparados! Agora vamos para o **Notebook 03: Avaliacao do Modelo**, onde vamos treinar, comparar e interpretar os resultados.

**Nayane Araujo** | [GitHub](https://github.com/Nayanearaujo)